In [1]:
# Local environment check - dynamically load local_setup if running locally in VSCode
try:
    dbutils
except NameError:
    import os, sys
    from pathlib import Path
    
    # Walk up to locate project root folder
    curr_path = Path(os.getcwd()).resolve()
    project_root = None
    for _ in range(5):
        if (curr_path / "local_setup.py").exists():
            project_root = curr_path
            break
        curr_path = curr_path.parent
        
    if project_root:
        sys.path.append(str(project_root))
        from local_setup import spark, dbutils, display
    else:
        print("Warning: Could not locate local_setup.py in parent directories!")


Initializing local Spark session with Delta Lake support...
:: loading settings :: url = jar:file:/home/mi/.pyenv/versions/3.10.14/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/mi/.ivy2/cache
The jars for the packages stored in: /home/mi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f45b78a0-6b53-4228-b23a-a64531ff246f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 171ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0

Spark session successfully created.


In [2]:
import os
import sys
import shutil
import json
from pathlib import Path

# Force local file system synchronization
os.sync()

# Dynamic workspace root directory resolution
ROOT_DIR = Path(os.getcwd()).parent
sys.path.append(str(ROOT_DIR))

# Define widgets for dynamic ClientContainer selection
try:
    dbutils.widgets.text("ClientContainer", "274", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "274"

26/06/29 16:24:11 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
from Shared.EDIProcessing import EDIProcessor, CSVConverter
from DimProvider.EDIProcessing.mapper import Mapper

In [4]:
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    
    # Overwrite if target exists to prevent pipeline blocks
    if target_path.exists():
        target_path.unlink()
        
    shutil.move(str(src_path), str(target_path))
    # Flush disk to ensure sync
    os.sync()
    return target_path

In [5]:
def process_single_file(incoming_file_path: Path, base_source_dir: Path, layout_id: str) -> tuple:
    """Parses an EDI file, extracts metadata, maps records, and generates a schema-compliant CSV."""
    active_file_path = incoming_file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")
        
        # Transition: pending -> inprogress
        active_file_path = move_file(active_file_path, base_source_dir / "inprogress")

        # Core Parsing & Domain Mapping
        structured_json = EDIProcessor().parse(str(active_file_path))
        
        # Metadata Extraction
        interchange = structured_json.get('interchange', {})
        client_id = interchange.get('sender_id', '02').strip()
        file_id = interchange.get('control_number', '01').strip()
        
        if not client_id:
            client_id = "02"
        if not file_id:
            file_id = "01"
            
        # Target CSV Location
        target_csv_name = f"{active_file_path.stem}.csv"
        target_csv_path = ROOT_DIR / "temp" / layout_id / target_csv_name
        target_csv_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Code Conversion Execution
        CSVConverter().converter(Mapper().map_provider(structured_json), str(target_csv_path))
            
        print(f"client id: {client_id}, file id: {file_id}, layout id: {layout_id}, csv name: {target_csv_name}")
        return client_id, file_id, layout_id, target_csv_name, active_file_path
        
    except Exception as e:
        print(f"Failed processing {active_file_path.name}: {e}")
        if active_file_path.exists():
            move_file(active_file_path, base_source_dir / "failed")
        raise

In [6]:
def build_payloads(processed_files: list) -> str:
    """Generates precise production payloads for downstream bronze ingest stages."""
    process_list = []
    
    for f in processed_files:
        specific_client_container = f"{ROOT_DIR}/temp/{f['layout_id']}"
        schema_file = "provider_hierarchy_7.12_schema.json"
        dest_folder = "provider_hierarchy"
        
        process_list.append({
            "ClientID": f['client_id'],
            "FileID": f['file_id'],
            "FileName": f['csv_filename'],
            "ClientContainer": specific_client_container,
            "CurrentFolderPath": "",
            "ProcessedFolderPath": f"/Volumes/{client_container}/bronze/processed_parquet/{dest_folder}",
            "ColumnDelimiter": ",",
            "HasHeader": "true",
            "IgnoreHeader": "False",
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "SchemaFileName": schema_file,
            "SchemaFilePath": f"{ROOT_DIR}/DimProvider/Bronze/Schema",
            "TextQualifier": "\""
        })
        
    return json.dumps({"FileIds": process_list})

In [7]:
def trigger_silver_notebooks(client_container_val: str):
    """Triggers Silver layer notebooks in the correct dependency order."""
    silver_notebooks_base = f"{ROOT_DIR}/DimProvider/Silver/Notebooks"
    
    try:
        # Create Provider Hierarchy table (exclusively for 274 project)
        print("\n=== Triggering ProviderHierarchy (Silver Layer) ===")
        dbutils.notebook.run(
            f"{silver_notebooks_base}/ProviderHierarchy", 
            600, 
            {"ClientContainer": client_container_val}
        )
        print("ProviderHierarchy completed successfully")
        
    except Exception as e:
        print(f"Silver layer processing failed: {e}")
        raise

In [8]:
def trigger_gold_notebooks(client_container_val: str):
    """Triggers Gold layer notebooks using the generic subgroup processor."""
    gold_notebooks_base = f"{ROOT_DIR}/DimProvider/Gold/Notebooks"
    
    try:
        # Load dimProvider (final merged dimension)
        print("\n=== Triggering dimProvider (Gold Layer) ===")
        dbutils.notebook.run(
            f"{gold_notebooks_base}/GenericSubGroupProcessing", 
            600, 
            {
                "ClientContainer": client_container_val,
                "SubGroupConfigPath": f"{ROOT_DIR}/DimProvider/Gold/Schema/dimProvider.json"
            }
        )
        print("dimProvider Gold load completed successfully")
        
    except Exception as e:
        print(f"Gold layer processing failed: {e}")
        raise

In [9]:
def main():
    # Detect pending EDI files
    base_274_dir = ROOT_DIR / "source/274"
    pending_274_dir = base_274_dir / "pending"
    
    processed_files = []
    
    # Check for 274 (Provider Hierarchy) pending files
    if pending_274_dir.exists():
        incoming_274 = [f for f in pending_274_dir.iterdir() if f.is_file() and not f.name.startswith('.')]
        for file_path in incoming_274:
            try:
                c_id, f_id, l_id, csv_filename, active_path = process_single_file(file_path, base_274_dir, "274")
                processed_files.append({
                    'client_id': c_id, 
                    'file_id': f_id, 
                    'layout_id': l_id,
                    'csv_filename': csv_filename,
                    'active_file_path': active_path
                })
            except Exception as e:
                print(f"Skipping 274 file {file_path.name}: {e}")

    if not processed_files:
        print("No pending EDI files found to process. Proceeding directly to Silver and Gold layers.")
        # Trigger Silver and Gold layers using existing Bronze data
        try:
            trigger_silver_notebooks(client_container)
            trigger_gold_notebooks(client_container)
            print("\nPipeline execution sequence completed successfully (Silver → Gold).")
        except Exception as e:
            print(f"Pipeline execution failed: {e}")
            raise
        return

    # If there are new CSVs, trigger Bronze Ingestion
    process_payload = build_payloads(processed_files)
    notebook_base = f"{ROOT_DIR}/Shared/Notebooks"
    
    try:
        print("\n=== Triggering FilesToProcess (Bronze Layer) ===")
        res_str = dbutils.notebook.run(f"{notebook_base}/FilesToProcess", 600, {"ProcessedJSON": process_payload})
        print(f"FilesToProcess completed with response: {res_str}")
        
        # Validate child notebook execution status
        try:
            res_json = json.loads(res_str)
            for run_res in res_json:
                if run_res.get("Status") != "SUCCESS":
                    raise Exception(f"Bronze ingestion failed for file {run_res.get('FileName')}: {run_res.get('ErrorMessage')}")
        except Exception as e:
            if "Bronze ingestion failed" in str(e):
                raise
            print(f"Warning: Could not parse response JSON: {e}")
        
        # Trigger Silver layer notebooks (Fixed to pass catalog)
        trigger_silver_notebooks(client_container)
        
        # Trigger Gold layer notebooks
        trigger_gold_notebooks(client_container)
        
        # Transition: inprogress -> processed ONLY after complete success
        for f in processed_files:
            move_file(f['active_file_path'], base_274_dir / "processed")
        
        print("\nPipeline execution sequence completed successfully (Bronze → Silver → Gold).")
    except Exception as e:
        print(f"Downstream orchestration failed: {e}")
        # Transition: inprogress -> failed on downstream failure
        for f in processed_files:
            if f['active_file_path'].exists():
                move_file(f['active_file_path'], base_274_dir / "failed")
        raise

if __name__ == "__main__":
    main()

Node is None for segment HL, using original
Node is None for segment NM1, using original
Node is None for segment PER, using original
Node is None for segment HL, using original
Node is None for segment NM1, using original
Node is None for segment N3, using original
Node is None for segment N4, using original
Node is None for segment REF, using original
Node is None for segment PER, using original
Node is None for segment HL, using original
Node is None for segment NM1, using original
Node is None for segment PRV, using original
Node is None for segment N3, using original
Node is None for segment N4, using original
Node is None for segment REF, using original
Node is None for segment REF, using original
Node is None for segment DMG, using original
Node is None for segment LUI, using original
Node is None for segment LUI, using original
Node is None for segment HSD, using original
Node is None for segment TPB, using original
Node is None for segment N1, using original
Node is None for s

client id: SENDERID, file id: 000000123, layout id: 274, csv name: provider_hierarchy_nonsolo.csv

=== Triggering FilesToProcess (Bronze Layer) ===

[LOCAL EXECUTION] Running child notebook: /home/mi/Desktop/claim processing/claimprocessing_provider274/Shared/Notebooks/FilesToProcess
Arguments: {'ProcessedJSON': '{"FileIds": [{"ClientID": "SENDERID", "FileID": "000000123", "FileName": "provider_hierarchy_nonsolo.csv", "ClientContainer": "/home/mi/Desktop/claim processing/claimprocessing_provider274/temp/274", "CurrentFolderPath": "", "ProcessedFolderPath": "/Volumes/274/bronze/processed_parquet/provider_hierarchy", "ColumnDelimiter": ",", "HasHeader": "true", "IgnoreHeader": "False", "FileLayoutID": "274", "FileLayoutDescription": "Standard274", "SchemaFileName": "provider_hierarchy_7.12_schema.json", "SchemaFilePath": "/home/mi/Desktop/claim processing/claimprocessing_provider274/DimProvider/Bronze/Schema", "TextQualifier": "\\""}]}'}
Loading notebook cells: /home/mi/Desktop/claim pro

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|FileIds                                                                                                                                                                                                                                                                                                                                                                                                                

26/06/29 16:24:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-06-29 16:24:50,013 - INFO - ============================================================
2026-06-29 16:24:50,014 - INFO - Provider Hierarchy Silver Layer Processing Started
2026-06-29 16:24:50,015 - INFO - Log file: /home/mi/Desktop/claim processing/claimprocessing_provider274/logs/provider_hierarchy_20260629_162450.log
2026-06-29 16:24:50,016 - INFO - ============================================================
2026-06-29 16:24:50,017 - INFO - Configuration loaded:
2026-06-29 16:24:50,018 - INFO -   Source Path: /Volumes/274/bronze/processed_parquet/provider_hierarchy
2026-06-29 16:24:50,019 - INFO -   Silver Hierarchy Table: `274`.silver.provider_hierarchy
2026-06-29 16:24:50,021 - INFO - MAIN EXECUTION STARTED
2026-06-29 16:24:50,209 - INFO - Data check: /Volumes/274/bronze/processed_parquet/

[{"FileID":"000000123","FileName":"provider_hierarchy_nonsolo.csv","FullFilePath":"/Volumes/274/bronze/processed_parquet/provider_hierarchy","CurrentJobId":"Undefined","Status":"SUCCESS","RecordCount":"1","ErrorMessage":""}]
[LOCAL EXIT] Notebook exited with response: [{"FileID":"000000123","FileName":"provider_hierarchy_nonsolo.csv","FullFilePath":"/Volumes/274/bronze/processed_parquet/provider_hierarchy","CurrentJobId":"Undefined","Status":"SUCCESS","RecordCount":"1","ErrorMessage":""}]
FilesToProcess completed with response: [{"FileID":"000000123","FileName":"provider_hierarchy_nonsolo.csv","FullFilePath":"/Volumes/274/bronze/processed_parquet/provider_hierarchy","CurrentJobId":"Undefined","Status":"SUCCESS","RecordCount":"1","ErrorMessage":""}]

=== Triggering ProviderHierarchy (Silver Layer) ===

[LOCAL EXECUTION] Running child notebook: /home/mi/Desktop/claim processing/claimprocessing_provider274/DimProvider/Silver/Notebooks/ProviderHierarchy
Arguments: {'ClientContainer': '274'

...

Loading Bronze Provider Hierarchy data...
Applying cleaning and normalization transformations...


Found 3 records to write
Writing to Silver Hierarchy Table: `274`.silver.provider_hierarchy


Silver Provider Hierarchy table created successfully!
ProviderHierarchy completed successfully

=== Triggering dimProvider (Gold Layer) ===

[LOCAL EXECUTION] Running child notebook: /home/mi/Desktop/claim processing/claimprocessing_provider274/DimProvider/Gold/Notebooks/GenericSubGroupProcessing
Arguments: {'ClientContainer': '274', 'SubGroupConfigPath': '/home/mi/Desktop/claim processing/claimprocessing_provider274/DimProvider/Gold/Schema/dimProvider.json'}
Loading notebook cells: /home/mi/Desktop/claim processing/claimprocessing_provider274/DimProvider/Gold/Notebooks/GenericSubGroupProcessing.ipynb
Client Container: new
Config Path: /home/mi/Desktop/claim processing/claimprocessing_provider274/DimProvider/Gold/Schema/dimProvider.json
Source: new.silver.provider_hierarchy → Destination: new.gold.dimprovider
Client Container: 274
Safe Catalog: `274`
Config Path: /home/mi/Desktop/claim processing/claimprocessing_provider274/DimProvider/Gold/Schema/dimProvider.json
Reading config from: 

  DestinationTable temp view created

  Executing Merge Script...


 Merge completed successfully for dimprovider

Processing Summary
+----------------------+-----------+-----+-------+
|destination           |entity     |error|status |
+----------------------+-----------+-----+-------+
|`274`.gold.dimprovider|dimprovider|     |SUCCESS|
+----------------------+-----------+-----+-------+


Processed 1 entities: 1 succeeded, 0 failed, 0 skipped
[LOCAL EXIT] Notebook exited with response: Processed 1 entities: 1 succeeded, 0 failed, 0 skipped
dimProvider Gold load completed successfully

Pipeline execution sequence completed successfully (Bronze → Silver → Gold).
